In [ ]:
# # Python
# from source_code.api_client import SimapAPIClient, build_search_filters
#
# client = SimapAPIClient()
#
# filters = build_search_filters(
#     start_date="2024-01-01",
#     end_date="2024-06-30",
#     types=["OB00", "OB01", "OB02"],  # nach Bedarf anpassen/ergänzen
#     # weitere optionale Filter:
#     # contract_types=["WORKS","SERVICES"],
#     # procedures=["OPEN","RESTRICTED"],
#     # cpv=["45000000"],
#     # bkp=["40","211"],
#     # keywords="*Brücke*",
#     # canton_codes=["ZH","BE"],
# )
#
# publications = client.iterate_publications(filters, records_per_page=1000)
# csv_path = client.export_publications_csv(publications, filename="auftraege_custom.csv")
# print(f"CSV gespeichert: {csv_path}")


In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import pandas as pd

df = pd.read_csv(r"C:\Users\Raguj\PycharmProjects\MLOps-HS-25\data\raw\CSV_Tableuprep_12_11.csv")


print(df.shape)
print(df.head())


df["text_attribute"] = df["Title"] + " " + df["Description"]
df["text_attribute"] = df["text_attribute"].str.lower()


df = df.dropna(subset=["Order_Type"])
df = df[df["Order_Type"] != "unknown"]
y = df["Order_Type"]
x = df[["text_attribute", "Country", "Canton", "Process_Type", "Projekt_Type"]].fillna("")



from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=42,stratify=y)


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Wenn zu recheintensiv > max_features abesetze
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=15000), "text_attribute"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Country", "Canton", "Process_Type", "Projekt_Type"]),
    ], remainder="drop"
)

from sklearn.ensemble import RandomForestClassifier

# Wenn zu lang dured > n_estimators abesetze
model = RandomForestClassifier(n_estimators=666, random_state=42)


from sklearn.pipeline import Pipeline

#Zum zemmefüehre vo td idf, onehot und random forest
klassifikator = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])

klassifikator.fit(X_train, y_train)


#Uswertig vo de Performance
from sklearn.metrics import classification_report

y_pred = klassifikator.predict(X_test)
print(classification_report(y_test, y_pred))

(1000, 24)
                             Projekt_id                        Publication_id  \
0  6204fbdd-5781-4adc-b548-ce2666c00e29  f5c94e20-08f7-4d20-b624-5512e85b59ed   
1  93f19304-08f6-4a48-9db9-fdf95a793a38  f601d09d-b79a-4573-b26c-491c30bf2ae6   
2  f92403e2-af49-4e3a-a4bb-26f394d57b42  4f3a90cd-31fd-4906-9af5-7b441bb946c5   
3  70c2b00e-5355-47ac-890d-46671b80ab58  253f719d-1e48-48fc-ac62-29f673fffae8   
4  2552a95e-3371-46d4-882f-090d2ce5836e  5edd535c-c357-40a6-a180-b206350d88c5   

  Publication_date Publication_type  \
0       2025-11-08            award   
1       2025-11-08      abandonment   
2       2025-11-08            award   
3       2025-11-08            award   
4       2025-11-08            award   

                                               Title  \
0  Rénovation de l'hôpital Le Samaritain en clini...   
1  CCF (Centrale Chaleur Force) Bois Déchets Lign...   
2  Rénovation de l'hôpital Le Samaritain en clini...   
3            Managed Security Operation Cen

In [80]:
# ==============================
# Modell-Vergleich für Simap-Projekt
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# === 1. CSV laden ===
df = pd.read_csv(r"C:\Users\Raguj\PycharmProjects\MLOps-HS-25\data\raw\simap_last10d_sample.csv")

print(df.shape)
print(df.head())



# === 2. Textspalte bauen ===
df["text_attribute"] = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()

# === 3. Nur gültige Zeilen behalten ===
df = df.dropna(subset=["order_type", "award_value"])
df = df[df["order_type"] != "unknown"]




# Schritt : Sonderzeichen, Währungen und Apostrophe entfernen
df["award_value_clean"] = (
    df["award_value"].astype(str)
    .str.replace("CHF", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace("_", "", regex=False)
    .str.replace(",", ".", regex=False)  # Komma → Punkt für Dezimalzahlen
)

# Schritt : Ungültige Einträge in NaN umwandeln
df["award_value_clean"] = pd.to_numeric(df["award_value_clean"], errors="coerce")


bins=[0, 100_000, 1_000_000, 10_000_000, float("inf")]
labels=["klein", "mittel", "gross", "sehr gross"]

df["value_category"] = pd.cut(
    df["award_value_clean"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# NaN-Zeilen entfernen
df = df.dropna(subset=["order_type", "value_category"])


# === 4. X und y definieren ===
X = df[["text_attribute", "country", "canton", "process_type", "project_type"]].fillna("")

X = X.astype(str)


# Zielvariable
y_raw = {
    "order_type": df["order_type"],
    "value_category": df["value_category"]
}

targets = {}
label_encoders = {}
for target_name, y in y_raw.items():
    le = LabelEncoder()
    targets[target_name] = le.fit_transform(y)
    label_encoders[target_name] = le

# === 5. Vorverarbeitung ===
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=15000), "text_attribute"),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["country", "canton", "process_type", "project_type"]),
    ],
    remainder="drop"
)

# === 6. Modelle definieren ===
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC()
}

# === 7. Training für beide Zielvariablen ===
for target_name, y in targets.items():
    print(f"\n==============================")
    print(f"Training für Zielvariable: {target_name}")
    print(f"==============================")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.1, random_state=42, stratify=y
    )
    for name, clf in models.items():
        print(f"\n Modell: {name}")
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)


        print(classification_report(y_test, y_pred))


(2337, 24)
                             project_id                        publication_id  \
0  acdb20e1-2210-460e-add0-4b93be8f4445  f2a6c8d8-f7da-4ed3-9700-3dc0829b8e07   
1  24b03a8d-eaed-407f-9ff8-4d6ba3687a35  752c355e-7201-4b6b-865c-1210965db091   
2  9ef225df-8677-4ed3-9b27-f53a7ac04172  5b02f5a8-e937-400c-a222-c720065bef90   
3  e64990e8-c4cd-47c3-8338-6795d725647c  3fdb1772-4cd1-48b4-a762-55ea5e75444d   
4  0d814cfd-0e1e-44ee-b3e7-c6fb5b112337  84cbd94d-2893-4252-a1ac-03c513a92e9b   

  publication_date pub_type  \
0       2025-11-12    award   
1       2025-11-12    award   
2       2025-11-12    award   
3       2025-11-12    award   
4       2025-11-12    award   

                                               title  \
0  0731.110 - Strada Nazionale N24 - Comune di St...   
1  Construction d'une nouvelle aile de salles de ...   
2  CHENEVIERS III - Systèmes de nettoyage faiscea...   
3  Renouvellement des lits de soins intensifs et ...   
4                                  